# S6.3 — Retrieval Node

Verifies the retrieval node for the agentic RAG LangGraph workflow.
The node invokes the advanced retrieval pipeline and converts SearchHits to SourceItems.

In [1]:
import sys
sys.path.insert(0, "../..")

## 1. Import & Verify Module Exists

In [2]:
from src.services.agents.nodes.retrieve_node import (
    ainvoke_retrieve_step,
    convert_search_hits_to_sources,
)
from src.services.agents.nodes import ainvoke_retrieve_step as exported_fn

assert ainvoke_retrieve_step is exported_fn, "Must be re-exported from __init__"
print("\u2713 retrieve_node module imports successfully")
print(f"  ainvoke_retrieve_step: {ainvoke_retrieve_step}")
print(f"  convert_search_hits_to_sources: {convert_search_hits_to_sources}")

✓ retrieve_node module imports successfully
  ainvoke_retrieve_step: <function ainvoke_retrieve_step at 0x10d8edc60>
  convert_search_hits_to_sources: <function convert_search_hits_to_sources at 0x10d8ed9e0>


## 2. SearchHit → SourceItem Conversion

In [3]:
from src.schemas.api.search import SearchHit
from src.services.agents.models import SourceItem

hits = [
    SearchHit(
        arxiv_id="1706.03762",
        title="Attention Is All You Need",
        authors=["Vaswani", "Shazeer"],
        score=0.95,
        chunk_text="The dominant sequence transduction models...",
        chunk_id="chunk-1",
    ),
    SearchHit(
        arxiv_id="1810.04805",
        title="BERT",
        authors=["Devlin"],
        score=0.88,
        chunk_text="We introduce BERT...",
        pdf_url="https://arxiv.org/pdf/1810.04805",
        chunk_id="chunk-2",
    ),
    SearchHit(arxiv_id="", title="No ID Paper", chunk_id="chunk-3"),
]

sources = convert_search_hits_to_sources(hits)

assert len(sources) == 2, f"Expected 2 (empty arxiv_id skipped), got {len(sources)}"
assert sources[0].arxiv_id == "1706.03762"
assert sources[0].url == "https://arxiv.org/abs/1706.03762"
assert sources[1].url == "https://arxiv.org/pdf/1810.04805"
assert all(isinstance(s, SourceItem) for s in sources)

print("\u2713 SearchHit → SourceItem conversion works correctly")
for s in sources:
    print(f"  [{s.arxiv_id}] {s.title} (score={s.relevance_score:.2f}) → {s.url}")

✓ SearchHit → SourceItem conversion works correctly
  [1706.03762] Attention Is All You Need (score=0.95) → https://arxiv.org/abs/1706.03762
  [1810.04805] BERT (score=0.88) → https://arxiv.org/pdf/1810.04805


## 3. Retrieve Step with Mock Pipeline

In [4]:
from unittest.mock import AsyncMock, MagicMock
from src.services.agents.context import AgentContext
from src.services.agents.state import create_initial_state
from src.services.retrieval.pipeline import RetrievalResult

# Build mock pipeline
mock_hits = [
    SearchHit(arxiv_id="1706.03762", title="Attention Is All You Need",
              authors=["Vaswani"], score=0.95, chunk_text="Self-attention...", chunk_id="c1"),
    SearchHit(arxiv_id="1810.04805", title="BERT",
              authors=["Devlin"], score=0.88, chunk_text="Pre-training...", chunk_id="c2"),
]
mock_result = RetrievalResult(
    results=mock_hits,
    query="What is attention?",
    stages_executed=["multi_query", "hybrid_search", "rerank"],
    total_candidates=15,
    timings={"multi_query": 0.2, "hybrid_search": 0.1, "rerank": 0.05},
)

pipeline = AsyncMock()
pipeline.retrieve = AsyncMock(return_value=mock_result)

context = AgentContext(llm_provider=MagicMock(), retrieval_pipeline=pipeline, top_k=5)
state = create_initial_state("What is attention?")

result = await ainvoke_retrieve_step(state, context)

assert len(result["sources"]) == 2
assert result["retrieval_attempts"] == 1
assert "retrieval" in result["metadata"]
meta = result["metadata"]["retrieval"]
assert meta["stages_executed"] == ["multi_query", "hybrid_search", "rerank"]
assert meta["total_candidates"] == 15
assert meta["num_results"] == 2

print("\u2713 ainvoke_retrieve_step works correctly")
print(f"  Sources: {len(result['sources'])}")
print(f"  Attempts: {result['retrieval_attempts']}")
print(f"  Stages: {meta['stages_executed']}")
print(f"  Candidates: {meta['total_candidates']}")

✓ ainvoke_retrieve_step works correctly
  Sources: 2
  Attempts: 1
  Stages: ['multi_query', 'hybrid_search', 'rerank']
  Candidates: 15


## 4. Edge Case: Pipeline is None

In [5]:
ctx_no_pipeline = AgentContext(llm_provider=MagicMock(), retrieval_pipeline=None)
state2 = create_initial_state("Test query")

result2 = await ainvoke_retrieve_step(state2, ctx_no_pipeline)

assert result2["sources"] == []
assert "error" in result2["metadata"]["retrieval"]
assert result2["retrieval_attempts"] == 1

print("\u2713 Graceful handling when pipeline is None")
print(f"  Error: {result2['metadata']['retrieval']['error']}")

Retrieval pipeline is None — cannot retrieve documents


✓ Graceful handling when pipeline is None
  Error: retrieval_pipeline is None


## 5. Edge Case: Pipeline Raises Exception

In [6]:
err_pipeline = AsyncMock()
err_pipeline.retrieve = AsyncMock(side_effect=RuntimeError("connection refused"))
ctx_err = AgentContext(llm_provider=MagicMock(), retrieval_pipeline=err_pipeline)
state3 = create_initial_state("Test query")

result3 = await ainvoke_retrieve_step(state3, ctx_err)

assert result3["sources"] == []
assert "error" in result3["metadata"]["retrieval"]

print("\u2713 Graceful handling when pipeline raises exception")
print(f"  Error: {result3['metadata']['retrieval']['error']}")

Retrieval pipeline failed: connection refused
Traceback (most recent call last):
  File "/Users/nishantgaurav/Project/PaperAlchemy/notebooks/specs/../../src/services/agents/nodes/retrieve_node.py", line 115, in ainvoke_retrieve_step
    retrieval_result = await context.retrieval_pipeline.retrieve(query, top_k=context.top_k)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/unittest/mock.py", line 2282, in _execute_mock_call
    raise effect
RuntimeError: connection refused


✓ Graceful handling when pipeline raises exception
  Error: connection refused


## Summary

S6.3 Retrieval Node verified:
- `ainvoke_retrieve_step` invokes pipeline and returns SourceItems
- `convert_search_hits_to_sources` maps fields correctly, skips empty arxiv_id
- Retrieval attempts incremented on each call
- Pipeline metadata stored in state
- Graceful degradation on None pipeline or exception